# 01. 추론(Predict) 기초 — 객체 탐지 깊게 파기

학습은 나중에. 먼저 **사전학습 모델로 추론**하면서 YOLO가 무엇을 어떻게 내놓는지 손에 익힙니다.

이 노트북에서 배우는 것
1. `predict()`의 핵심 인자 (`conf`, `iou`, `imgsz`, `classes` ...)
2. `Results` / `Boxes` 객체 완전 분해
3. 결과 시각화·저장
4. 여러 장 배치 추론
5. 탐지 결과를 표(structured data)로 뽑아내기

> 커널이 **Python (ultralytics-practice)** 인지 확인하고 시작하세요.

## 0. 준비

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
from ultralytics import YOLO
from PIL import Image

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)

model = YOLO("yolo26n.pt")   # 탐지용 사전학습 모델
IMG = "https://ultralytics.com/images/bus.jpg"

## 1. 가장 기본 추론

`predict()`는 이미지마다 하나씩 `Results` 객체를 담은 **리스트**를 반환합니다. 한 장을 넣어도 리스트예요.

In [ ]:
results = model.predict(IMG, device=DEVICE)
print("반환 타입:", type(results), "| 길이:", len(results))

r = results[0]
print("탐지 개수:", len(r.boxes))
print("추론 속도(ms):", r.speed)   # preprocess / inference / postprocess

## 2. 핵심 인자들 — 하나씩 바꿔보며 체감하기

| 인자 | 의미 | 기본값 |
|---|---|---|
| `conf` | 이 신뢰도 미만 탐지는 버림 (낮출수록 더 많이 잡음) | 0.25 |
| `iou`  | NMS 겹침 임계값 (낮출수록 겹친 박스 더 많이 제거) | 0.7 |
| `imgsz`| 추론 입력 크기 (클수록 작은 물체 유리, 느림) | 640 |
| `classes` | 특정 클래스만 남기기 (예: 사람=0) | None |
| `max_det` | 이미지당 최대 탐지 수 | 300 |

In [ ]:
# conf 를 바꿔가며 탐지 개수 변화 관찰
for c in [0.05, 0.25, 0.5, 0.8]:
    n = len(model.predict(IMG, device=DEVICE, conf=c, verbose=False)[0].boxes)
    print(f"conf={c:<5} -> 탐지 {n}개")

In [ ]:
# 특정 클래스만 남기기 — 사람(0)만 탐지
r_person = model.predict(IMG, device=DEVICE, classes=[0], verbose=False)[0]
print("사람만:", len(r_person.boxes), "개")
print("검출된 클래스 id:", r_person.boxes.cls.tolist())

## 3. Results / Boxes 객체 완전 분해

탐지 정보는 전부 `r.boxes` 안에 있습니다. 좌표는 세 가지 형식으로 제공돼요.

| 속성 | 뜻 |
|---|---|
| `boxes.xyxy` | `[x1,y1,x2,y2]` 좌상단·우하단 (픽셀) |
| `boxes.xywh` | `[중심x, 중심y, 너비, 높이]` (픽셀) |
| `boxes.xyxyn` | xyxy를 0~1로 정규화 (이미지 크기 무관) |
| `boxes.conf` | 신뢰도 |
| `boxes.cls` | 클래스 id (실수 텐서 → int 변환 필요) |
| `r.names` | id→이름 딕셔너리 |

In [ ]:
r = model.predict(IMG, device=DEVICE, verbose=False)[0]
b = r.boxes

print("xyxy[0] :", [round(v,1) for v in b.xyxy[0].tolist()])
print("xywh[0] :", [round(v,1) for v in b.xywh[0].tolist()])
print("xyxyn[0]:", [round(v,3) for v in b.xyxyn[0].tolist()])
print("conf[0] :", round(float(b.conf[0]), 3))
print("cls[0]  :", int(b.cls[0]), "->", r.names[int(b.cls[0])])

## 4. 결과 시각화하기

`r.plot()`은 박스·라벨이 그려진 이미지를 **numpy 배열(BGR)** 로 돌려줍니다. 노트북에서 보려면 RGB로 뒤집으면 돼요.

In [ ]:
im_bgr = r.plot()
Image.fromarray(im_bgr[..., ::-1])   # BGR -> RGB

In [ ]:
# 파일로 저장하고 싶다면 predict 에서 save=True
saved = model.predict(IMG, device=DEVICE, save=True, verbose=False)
print("저장 폴더:", saved[0].save_dir)

## 5. 여러 장 배치 추론

리스트/폴더/URL 여러 개를 한 번에 넣으면 결과도 순서대로 리스트로 나옵니다.

In [ ]:
imgs = [
    "https://ultralytics.com/images/bus.jpg",
    "https://ultralytics.com/images/zidane.jpg",
]
batch = model.predict(imgs, device=DEVICE, verbose=False)

for i, res in enumerate(batch):
    print(f"이미지 {i}: {len(res.boxes)}개 탐지")

## 6. 탐지 결과를 표(DataFrame)로 뽑기

실무에서 자주 하는 작업: 탐지 결과를 순회하며 구조화된 데이터로 정리.

In [ ]:
import pandas as pd

r = model.predict(IMG, device=DEVICE, verbose=False)[0]
rows = []
for i in range(len(r.boxes)):
    x1, y1, x2, y2 = r.boxes.xyxy[i].tolist()
    rows.append({
        "class": r.names[int(r.boxes.cls[i])],
        "conf": round(float(r.boxes.conf[i]), 3),
        "x1": round(x1), "y1": round(y1),
        "x2": round(x2), "y2": round(y2),
    })

pd.DataFrame(rows).sort_values("conf", ascending=False)

## 7. 연습문제 (직접 코드 채워보기)

1. `zidane.jpg`에서 **사람만** 탐지해 개수를 출력해 보세요.
2. `conf=0.4, iou=0.5`로 `bus.jpg`를 추론하고 결과 이미지를 노트북에 표시해 보세요.
3. 배치 추론 결과 전체를 하나의 DataFrame으로 합쳐 보세요. (이미지 번호 컬럼 추가)

아래 빈 셀에서 연습하세요.

In [ ]:
# 연습 공간


---
### 정리
- `predict()` → `Results` 리스트, 핵심 정보는 `r.boxes`
- `conf`/`iou`/`classes`로 탐지 결과를 제어
- `xyxy`·`xywh`·`xyxyn` 세 가지 좌표 형식
- `r.plot()`으로 시각화, `save=True`로 저장

**다음 노트북 →** `02_datasets_yaml` : 데이터셋 구조와 YAML, 커스텀 데이터 준비